# fedOPD — what is actually in the data

Run after `fetch_dataset.ipynb`. Every number and every chart below is computed from the files on disk in this notebook. Nothing is quoted from documentation or estimated.

## The three datasets

**1. RTT monthly full extracts — `data/rtt_csv/`, 96 files, Apr 2018 – Mar 2026.**
The Referral to Treatment return. One row is a combination of *provider × commissioner × treatment function × RTT part type*, for one month. It is a count table, not patient records: every value is a number of pathways. This single table carries both signals the project needs — new referrals (the demand signal, `Part_3`) and the waiting-list state (`Part_1A`, `Part_1B`, `Part_2`, `Part_2A`).

**2. Provider reference — `data/reference/`.**
Built from the NHS Organisation Data Service. `trust_icb.csv` maps each NHS trust to its Integrated Care Board and postcode. `provider_types.csv` (built below) classifies *every* organisation appearing in the RTT files by its ODS primary role, which is what separates NHS trusts from independent-sector providers.

**3. KH03 overnight beds — `data/kh03/`, quarterly workbooks.**
Average beds available and occupied per provider per quarter. Same provider codes, so it joins onto the RTT table. Used only for the phase-11 bed-context extension.

## What this notebook establishes

1. File inventory, sizes, naming, and the schema change
2. The anatomy of the RTT table: part types, value columns, treatment functions
3. **`C_999` is an aggregate row** — the single most important thing to know before summing anything
4. Providers: NHS vs independent, by count and by volume, measured across all 96 months
5. The organisational hierarchy: ICBs, commissioners, trusts
6. Departments (treatment functions) and how the set changes over time
7. KH03 contents
8. Deletion of the redundant archives

In [ ]:
import re                                       # parse month + naming template out of filenames
import collections                               # Counter for tallies
from pathlib import Path                         # filesystem paths
from datetime import datetime                    # build sortable dates from 'OCTOBER' + '2019'

import requests                                  # ODS API, for classifying providers
import pandas as pd                              # all tabular work
import matplotlib.pyplot as plt                  # charts

DATA = Path("../data")
RTT_ZIPS = DATA / "rtt_zips"                     # archives (deleted in the last section)
RTT_CSV = DATA / "rtt_csv"                       # the 96 extracted monthly files
REF = DATA / "reference"                         # lookup tables we build
KH03 = DATA / "kh03"                             # bed workbooks
HEADERS = {"User-Agent": "Mozilla/5.0"}          # ODS/nhs.uk reject the default python UA

MONTHS = "JANUARY FEBRUARY MARCH APRIL MAY JUNE JULY AUGUST SEPTEMBER OCTOBER NOVEMBER DECEMBER".split()

def month_of(name):
    """'RTT-OCTOBER-2019-full-extract.csv' -> datetime(2019, 10, 1)."""
    m = re.search(r"(" + "|".join(MONTHS) + r")-(\d{4})", name.upper())   # month word + year
    return datetime(int(m.group(2)), MONTHS.index(m.group(1)) + 1, 1)     # index+1 -> month number

csvs = sorted(RTT_CSV.glob("*.csv"), key=lambda p: month_of(p.name))      # chronological order
print(len(csvs), "monthly files:", month_of(csvs[0].name).date(), "->", month_of(csvs[-1].name).date())

## 1. File inventory

Size, naming style and month for every file. NHS renamed these repeatedly over eight years, so the naming templates are worth seeing explicitly — any globbing code has to survive them.

In [ ]:
rows = []
for p in csvs:
    header = open(p, newline="").readline().rstrip("\r\n")               # first line only — cheap
    cols = tuple(c.strip().strip('"') for c in header.split(","))        # strip quotes and spaces
    rows.append({
        "month": month_of(p.name),
        "file": p.name,
        "mb": p.stat().st_size / 1e6,                                    # size on disk
        "n_cols": len(cols),                                             # schema width
        "date_prefix": bool(re.match(r"^\d{8}-", p.name)),               # '20191031-...' style?
        "revised": "revised" in p.name.lower(),                          # NHS reissued this month
    })

files = pd.DataFrame(rows)                                               # already chronological
headers = {p.name: tuple(c.strip().strip('"')
                         for c in open(p, newline="").readline().rstrip("\r\n").split(","))
           for p in csvs}                                                # filename -> column tuple

print(f"files            : {len(files)}")
print(f"total size       : {files.mb.sum() / 1000:.2f} GB")
print(f"size range       : {files.mb.min():.1f} - {files.mb.max():.1f} MB")
print(f"with date prefix : {int(files.date_prefix.sum())}   without: {int((~files.date_prefix).sum())}")
print(f"marked 'revised' : {int(files.revised.sum())}")
files.head()

In [ ]:
# every month present exactly once? MS = month start, so this is the exact expected 96-month set
expected = pd.date_range("2018-04-01", "2026-03-01", freq="MS")
print("expected months :", len(expected))
print("distinct months :", files.month.nunique())
print("missing         :", [d.strftime('%b%Y') for d in expected.difference(files.month)] or "none")
print("duplicated      :", files.month.value_counts().loc[lambda s: s > 1].to_dict() or "none")

In [ ]:
def template(name):
    """Collapse a filename to its naming template so the distinct styles can be counted."""
    s = re.sub(r"^\d{8}-", "<DATE>-", name)                              # leading yyyymmdd stamp
    s = re.sub(r"(" + "|".join(MONTHS) + r")", "<MONTH>", s.upper())     # OCTOBER -> <MONTH>
    return re.sub(r"\d{4}", "<YEAR>", s)                                 # 2019 -> <YEAR>

files["template"] = files.file.map(template)
# when was each naming style in use? min/max show the changeover points
print(files.groupby("template").month.agg(n="count", first="min", last="max").to_string())

## 2. The schema change in April 2021

The column count is not constant across the eight years. This is the single largest structural break in the data and it silently misaligns any code that assumes a fixed column list.

In [ ]:
print("distinct column counts:", files.n_cols.value_counts().to_dict())
# find the exact month each schema width starts
prev = None
for r in files.itertuples(index=False):
    if r.n_cols != prev:                                                 # width changed here
        print(f"  {r.n_cols:3d} columns start {r.month:%b %Y}  ({r.file})")
        prev = r.n_cols

# what the extra columns actually are: compare one file of each width
narrow = headers[files[files.n_cols == files.n_cols.min()].file.iloc[0]]  # 69-col example
wide = headers[files[files.n_cols == files.n_cols.max()].file.iloc[0]]    # 121-col example
added = [c for c in wide if c not in narrow]                             # present only in the wide schema
dropped = [c for c in narrow if c not in wide]                           # present only in the narrow one
print(f"\ncolumns added in the wider schema : {len(added)}")
print("  first 5:", added[:5])
print("  last 5 :", added[-5:])
print(f"columns removed                   : {dropped}")

**Read the output above.** The single `Gt 52 Weeks SUM 1` bucket is replaced from April 2021 by weekly bands running 52→104 weeks. Nothing else changes.

Consequence for phases 2 and 4: a long-wait series spanning the break must sum the weekly bands back into one >52-week figure from April 2021 onward. The earlier period cannot be disaggregated to match, so the coarser representation governs.

## 3. Anatomy of the RTT table

What one monthly file contains: the five RTT part types, and which column actually holds the value for each.

In [ ]:
latest = csvs[-1]                                                        # most recent month
df = pd.read_csv(latest, low_memory=False)                               # full file, one month
print(latest.name, "->", f"{len(df):,} rows x {len(df.columns)} columns\n")

# the five part types and what each one counts
print(df.groupby(["RTT Part Type", "RTT Part Description"]).size().rename("rows").to_string())

In [ ]:
# which numeric column carries the value for each part type?
# 'Total All' is the row total; 'Total' is populated only for the waiting-time-banded parts
for pt in sorted(df["RTT Part Type"].unique()):
    d = df[df["RTT Part Type"] == pt]
    banded = [c for c in d.columns if "Weeks SUM" in c and d[c].fillna(0).abs().sum() > 0]
    print(f"{pt:8s} Total All={d['Total All'].fillna(0).sum():>12,.0f}   "
          f"Total={d['Total'].fillna(0).sum():>12,.0f}   populated week-bands={len(banded)}")

`Part_3` (new RTT periods) is the **demand signal** the forecaster predicts: it has no waiting-time bands, only a single `Total All`. The other four parts describe the waiting list and carry the week-band breakdown used by the phase-4 flow state.

## 4. `C_999` is an aggregate row, not a department

The `Treatment Function Code` column contains a code named "Total". It is **not** a specialty — it is the sum of all the others, published as extra rows in the same table. Summing the column without excluding it double-counts everything.

This is proved numerically below rather than assumed.

In [ ]:
AGG = "C_999"                                                            # the aggregate code

for pt in ["Part_3", "Part_2"]:                                          # demand, then waiting list
    d = df[df["RTT Part Type"] == pt]
    is_agg = d["Treatment Function Code"].eq(AGG)                        # True for the C_999 rows
    total_rows = d.loc[is_agg, "Total All"].sum()                        # what C_999 claims
    other_rows = d.loc[~is_agg, "Total All"].sum()                       # sum of the real specialties
    print(f"{pt}:  C_999 = {total_rows:>12,.0f}   sum of specialties = {other_rows:>12,.0f}   "
          f"difference = {total_rows - other_rows:,.0f}")

    # per-provider, not just nationally — a national match could hide offsetting errors
    per = d.pivot_table(index="Provider Org Code", columns=is_agg.map({True: "agg", False: "parts"}),
                        values="Total All", aggfunc="sum").fillna(0)
    mismatched = (per["agg"] - per["parts"]).abs().gt(0.5).sum()         # tolerance for float noise
    print(f"         providers where C_999 != sum(specialties): {mismatched} of {len(per)}\n")

In [ ]:
# the practical size of the trap: the England waiting list, computed both ways
d2 = df[df["RTT Part Type"] == "Part_2"]                                 # incomplete pathways = the waiting list
naive = d2["Total All"].sum()                                            # forgetting to exclude C_999
correct = d2[d2["Treatment Function Code"] != AGG]["Total All"].sum()     # specialties only
print(f"naive sum including C_999 : {naive:>14,.0f}")
print(f"correct (specialties only): {correct:>14,.0f}")
print(f"inflation factor          : {naive / correct:.2f}x")

## 5. Departments — the treatment functions

Each treatment function is a department, and is the unit the per-service agent acts on. The set is not constant over the eight years.

In [ ]:
tf = df[["Treatment Function Code", "Treatment Function Name"]].drop_duplicates().sort_values("Treatment Function Code")
print(f"codes in {latest.name}: {len(tf)}  (of which {len(tf) - 1} are real departments, 1 is the C_999 total)\n")
print(tf.to_string(index=False))

In [ ]:
# demand per department this month, C_999 excluded, largest first
dem = df[(df["RTT Part Type"] == "Part_3") & (df["Treatment Function Code"] != AGG)]
by_tf = (dem.groupby("Treatment Function Name")["Total All"].sum()
            .sort_values(ascending=False).rename("new_referrals"))
share = (by_tf / by_tf.sum() * 100).round(1).rename("pct")               # share of national demand
print(pd.concat([by_tf.map('{:,.0f}'.format), share], axis=1).to_string())

## 6. Providers — NHS vs independent sector

The RTT return covers every provider of NHS-funded elective care, which includes private hospitals and chains doing NHS work. They are not NHS trusts and are not part of the trust-level demand cycle the project models, so they have to be identified explicitly rather than assumed away.

Classification is by **ODS primary role**, looked up per organisation — not guessed from the code format.

In [ ]:
# collect every distinct provider that appears anywhere in the 96 months
seen, appearances = {}, collections.Counter()
for p in csvs:
    d = pd.read_csv(p, usecols=["Provider Org Code", "Provider Org Name"],
                    low_memory=False).drop_duplicates()                  # 2 columns only — fast
    for code, name in d.itertuples(index=False):
        seen.setdefault(code, name)                                      # keep the first name seen
        appearances[code] += 1                                           # how many months it appears in

providers = pd.DataFrame({"code": list(seen), "name": list(seen.values())})
providers["n_months"] = providers.code.map(appearances)                  # 1..96
print(f"distinct providers across all 96 months: {len(providers)}")
print("code length:", providers.code.str.len().value_counts().to_dict())

In [ ]:
# look each one up in ODS to get its primary role; cache to disk so this runs once
types_path = REF / "provider_types.csv"
ORD = "https://directory.spineservices.nhs.uk/ORD/2-0-0"

if types_path.exists():
    ptypes = pd.read_csv(types_path)                                     # already built
else:
    roles = {r["id"]: r["displayName"]                                   # role code -> readable name
             for r in requests.get(f"{ORD}/roles", headers=HEADERS).json()["Roles"]}
    out = []
    for r in providers.itertuples(index=False):
        resp = requests.get(f"{ORD}/organisations/{r.code}", headers=HEADERS)
        if resp.status_code != 200:                                      # code not in ODS at all
            out.append({"code": r.code, "name": r.name, "n_months": r.n_months,
                        "primary_role": "", "role_name": "NOT IN ODS", "status": ""})
            continue
        o = resp.json()["Organisation"]
        prim = [x["id"] for x in o["Roles"]["Role"] if x.get("primaryRole")]   # the primary role code
        pr = prim[0] if prim else ""
        out.append({"code": r.code, "name": r.name, "n_months": r.n_months,
                    "primary_role": pr, "role_name": roles.get(pr, "?"), "status": o["Status"]})
    ptypes = pd.DataFrame(out)
    ptypes.to_csv(types_path, index=False)

print(ptypes.groupby(["primary_role", "role_name"]).size().sort_values(ascending=False).rename("providers").to_string())

In [ ]:
# RO197 = NHS Trust, RO107 = Care Trust -> public NHS bodies; everything else is not
NHS_ROLES = {"RO197", "RO107"}
ptypes["sector"] = ptypes.primary_role.isin(NHS_ROLES).map({True: "NHS", False: "Independent"})
SECTOR = dict(zip(ptypes.code, ptypes.sector))                           # code -> 'NHS'/'Independent'

print(ptypes.groupby("sector").agg(providers=("code", "size"),
                                   median_months_present=("n_months", "median")).to_string())
print()
print("NHS bodies by role:")
print(ptypes[ptypes.sector == "NHS"].groupby("role_name").size().rename("n").to_string())
print()
print("Non-NHS by role:")
print(ptypes[ptypes.sector == "Independent"].groupby("role_name").size().rename("n").to_string())

### Provider count is the wrong denominator — measure the volume

Independent providers are numerous but individually tiny. What matters for the project is how much *demand* they account for.

In [ ]:
# one pass over all 96 months, reading only the 5 columns needed, aggregating as we go
USE = ["Provider Org Code", "Provider Parent Org Code", "RTT Part Type",
       "Treatment Function Code", "Total All"]

per_month, per_provider, per_tf = [], [], []
for p in csvs:
    mth = month_of(p.name)
    d = pd.read_csv(p, usecols=USE, low_memory=False)
    real = d[d["Treatment Function Code"] != AGG]                        # drop the aggregate rows once
    dem = real[real["RTT Part Type"] == "Part_3"]                        # demand only

    per_month.append({
        "month": mth,
        "rows": len(d),
        "providers": d["Provider Org Code"].nunique(),                   # distinct providers reporting
        "icbs": d["Provider Parent Org Code"].nunique(),                 # distinct parent organisations
        "tf_codes": real["Treatment Function Code"].nunique(),           # departments, C_999 excluded
        "demand": dem["Total All"].sum(),                                # national new referrals
    })
    g = dem.groupby("Provider Org Code")["Total All"].sum().reset_index()
    g["month"] = mth
    per_provider.append(g)                                               # month x provider demand
    t = dem.groupby("Treatment Function Code")["Total All"].sum().reset_index()
    t["month"] = mth
    per_tf.append(t)                                                     # month x department demand

monthly = pd.DataFrame(per_month)
prov_month = pd.concat(per_provider, ignore_index=True)
tf_month = pd.concat(per_tf, ignore_index=True)
prov_month["sector"] = prov_month["Provider Org Code"].map(SECTOR)       # attach NHS / Independent

monthly.to_csv(REF / "monthly_summary.csv", index=False)                 # cache for later phases
prov_month.to_csv(REF / "provider_month_demand.csv", index=False)
print(monthly.head().to_string(index=False))
print(f"\ntotal new referrals, all 96 months: {monthly.demand.sum():,.0f}")

In [ ]:
# the headline split, over the whole 8 years and in the latest month
for label, sub in [("all 96 months", prov_month),
                   (f"latest month ({monthly.month.max():%b %Y})",
                    prov_month[prov_month.month == prov_month.month.max()])]:
    g = sub.groupby("sector").agg(providers=("Provider Org Code", "nunique"),
                                  demand=("Total All", "sum"))
    g["provider_pct"] = (g.providers / g.providers.sum() * 100).round(1)
    g["demand_pct"] = (g.demand / g.demand.sum() * 100).round(1)
    g["demand"] = g.demand.map('{:,.0f}'.format)
    print(label); print(g.to_string(), "\n")

In [ ]:
# how the split moves over time — counts on the left, share of demand on the right
cnt = prov_month.groupby(["month", "sector"])["Provider Org Code"].nunique().unstack().fillna(0)
vol = prov_month.groupby(["month", "sector"])["Total All"].sum().unstack().fillna(0)
vol_pct = vol.div(vol.sum(axis=1), axis=0) * 100                         # row-normalised to %

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
cnt.plot(ax=ax[0], lw=1.5)
ax[0].set_title("Providers reporting, by sector"); ax[0].set_ylabel("providers"); ax[0].set_xlabel("")
vol_pct.plot(ax=ax[1], lw=1.5)
ax[1].set_title("Share of new referrals, by sector"); ax[1].set_ylabel("% of demand"); ax[1].set_xlabel("")
plt.tight_layout()

print("independent-sector share of demand (%):")
print(vol_pct["Independent"].describe().round(2).to_string())

In [ ]:
# size distribution: how big is a typical NHS trust vs a typical independent provider?
last = prov_month[prov_month.month == prov_month.month.max()]
print(f"monthly new referrals per provider, {prov_month.month.max():%b %Y}")
print(last.groupby("sector")["Total All"].describe().round(0).to_string())

## 7. The organisational hierarchy

The agentic layer escalates trust → ICB → national, so the real hierarchy has to be counted, not assumed. `Provider Parent Org Code` in the RTT file is the provider's parent organisation; `trust_icb.csv` is the independent confirmation from ODS.

In [ ]:
print(f"distinct provider-parent organisations in {latest.name}: {df['Provider Parent Org Code'].nunique()}")
print(f"distinct commissioner parents                          : {df['Commissioner Parent Org Code'].nunique()}")
print(f"distinct commissioners                                 : {df['Commissioner Org Code'].nunique()}")
print()
# how many NHS trusts sit under each parent, and how much demand each parent covers
nhs_codes = set(ptypes[ptypes.sector == "NHS"].code)
d3 = df[(df["RTT Part Type"] == "Part_3") & (df["Treatment Function Code"] != AGG)
        & df["Provider Org Code"].isin(nhs_codes)]
by_icb = d3.groupby(["Provider Parent Org Code", "Provider Parent Name"]).agg(
    trusts=("Provider Org Code", "nunique"), demand=("Total All", "sum")).sort_values("demand", ascending=False)
print(f"NHS trusts per ICB: min {by_icb.trusts.min()}, median {by_icb.trusts.median():.0f}, max {by_icb.trusts.max()}")
print()
print(by_icb.head(10).to_string())

In [ ]:
# the ODS reference table, for comparison
ref = pd.read_csv(REF / "trust_icb.csv")
print(f"trust_icb.csv rows        : {len(ref)}")
print(f"  status Active           : {(ref.status == 'Active').sum()}")
print(f"  with an ICB code        : {ref.icb_code.notna().sum()}")
print(f"  distinct ICBs           : {ref.icb_code.nunique()}")
print(f"  with a postcode         : {ref.postcode.notna().sum()}")
print()
# how many of the trusts actually reporting in the latest month are covered by the reference
reporting = set(d3["Provider Org Code"])
print(f"NHS trusts reporting in {latest.name}: {len(reporting)}")
print(f"  present in trust_icb.csv          : {len(reporting & set(ref.trust_code))}")

## 8. Demand over time, and the COVID shock

The series the forecaster has to be robust to.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(monthly.month, monthly.demand / 1e6, marker="o", ms=3, lw=1)
ax[0].set_title("National new RTT referrals per month"); ax[0].set_ylabel("millions")
ax[1].plot(monthly.month, monthly.providers, marker="o", ms=3, lw=1, label="providers")
ax[1].plot(monthly.month, monthly.tf_codes, marker="s", ms=3, lw=1, label="departments")
ax[1].set_title("Reporting entities per month"); ax[1].legend()
plt.tight_layout()

trough = monthly.loc[monthly.demand.idxmin()]                            # the covid collapse
peak = monthly.loc[monthly.demand.idxmax()]
print(f"lowest  month: {trough.month:%b %Y}  {trough.demand:,.0f} referrals")
print(f"highest month: {peak.month:%b %Y}  {peak.demand:,.0f} referrals")
print(f"trough is {trough.demand / monthly.demand.median() * 100:.0f}% of the 96-month median")

In [ ]:
# the department set is not constant either — when does each treatment function appear?
span = tf_month.groupby("Treatment Function Code").month.agg(["min", "max", "count"])
print("departments present in all 96 months:", int((span["count"] == 96).sum()), "of", len(span))
print()
print(span[span["count"] < 96].to_string())

## 9. KH03 overnight beds

In [ ]:
kh = sorted(KH03.glob("*"))
print(f"workbooks: {len(kh)}   total {sum(f.stat().st_size for f in kh) / 1e6:.1f} MB")
print("extensions:", collections.Counter(f.suffix.lower() for f in kh))
# the quarter each file covers is in its name, in several different formats
print("\nearliest/latest by name:")
print(" ", kh[0].name)
print(" ", kh[-1].name)